# 🔌 افزودن توزیع سفارشی در Climatology Engine

این نوت‌بوک نحوه افزودن یک توزیع آماری جدید به سیستم را نشان می‌دهد.

**مواردی که یاد می‌گیرید:**
- معماری پلاگین توزیع‌ها
- ساختار کلاس `DistributionPlugin`
- پیاده‌سازی تابع `fit()` برای توزیع جدید
- افزودن توزیع Weibull به عنوان مثال
- ثبت توزیع جدید در سیستم
- تست توزیع جدید روی داده نمونه
- مقایسه توزیع جدید با سایر توزیع‌ها

---

## 📐 معماری پلاگین توزیع‌ها

سیستم از معماری پلاگین استفاده می‌کند. هر توزیع جدید باید:

1. در پوشه `plugins/distributions/` قرار گیرد
2. از کلاس `DistributionPlugin` ارث‌بری کند
3. متد `fit()` را پیاده‌سازی کند
4. دارای نام، کد، پارامترها و تعداد پارامتر باشد

### ساختار کلاس پایه

```python
class DistributionPlugin:
    name = None          # نام توزیع
    code = None          # کد عددی منحصر‌به‌فرد
    params = []          # نام پارامترها
    n_params = 0         # تعداد پارامترها
    
    def fit(self, data):
        # پیاده‌سازی برازش
        return {'p1': val1, 'p2': val2, 'loglik': ..., 'aicc': ..., 'bic': ...}
```

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from core.engine.distribution_plugin import DistributionPlugin
from core.engine.plugin_loader import load_plugins

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ کتابخانه‌ها بارگذاری شدند.')

In [ ]:
# ============================================================================
# ۱. تعریف توزیع سفارشی: Weibull
# ============================================================================

class WeibullDistribution(DistributionPlugin):
    """
    توزیع Weibull
    f(x) = (shape/scale) * (x/scale)^(shape-1) * exp(-(x/scale)^shape)
    """
    name = "Weibull"
    code = 5  # کد جدید (بعد از 0-4)
    params = ["shape", "scale"]
    n_params = 2
    supports_negative = False
    supports_zero = True
    supports_positive = True
    extreme_only = False

    def fit(self, data):
        """
        برازش توزیع Weibull با روش حداکثر درست‌نمایی
        """
        # حذف مقادیر نامعتبر
        data = np.array(data)[~np.isnan(data)]
        if len(data) < 3:
            return {
                'shape': np.nan,
                'scale': np.nan,
                'loglik': np.nan,
                'aicc': np.nan,
                'bic': np.nan
            }
        
        # اطمینان از مثبت بودن داده
        if np.any(data <= 0):
            data = data - np.min(data) + 1e-6
        
        try:
            # برازش با استفاده از scipy
            shape, loc, scale = stats.weibull_min.fit(data, floc=0)
            
            # محاسبه log-likelihood
            loglik = np.sum(stats.weibull_min.logpdf(data, shape, loc=0, scale=scale))
            
            n = len(data)
            k = self.n_params
            aicc = -2*loglik + 2*k + (2*k*(k+1))/(n-k-1) if n > k+1 else np.inf
            bic = -2*loglik + k*np.log(n)
            
            return {
                'shape': shape,
                'scale': scale,
                'loglik': loglik,
                'aicc': aicc,
                'bic': bic
            }
        except Exception as e:
            return {
                'shape': np.nan,
                'scale': np.nan,
                'loglik': np.nan,
                'aicc': np.nan,
                'bic': np.nan
            }

    def pdf(self, x, params):
        """تابع چگالی احتمال"""
        shape = params.get('shape', 1.0)
        scale = params.get('scale', 1.0)
        return stats.weibull_min.pdf(x, shape, loc=0, scale=scale)

print("✅ توزیع Weibull تعریف شد.")
print(f"   نام: {WeibullDistribution.name}")
print(f"   کد: {WeibullDistribution.code}")
print(f"   پارامترها: {WeibullDistribution.params}")

In [ ]:
# ============================================================================
# ۲. تست توزیع Weibull روی داده مصنوعی
# ============================================================================

# تولید داده از توزیع Weibull
np.random.seed(42)
shape_true = 2.0
scale_true = 10.0
synthetic_data = stats.weibull_min.rvs(shape_true, loc=0, scale=scale_true, size=500)

print(f"📊 داده مصنوعی Weibull:")
print(f"   Shape واقعی: {shape_true}")
print(f"   Scale واقعی: {scale_true}")
print(f"   تعداد داده‌ها: {len(synthetic_data)}")

# برازش توزیع Weibull روی داده
weibull_dist = WeibullDistribution()
result = weibull_dist.fit(synthetic_data)

print("\n📈 نتایج برازش:")
for key, value in result.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

In [ ]:
# رسم هیستوگرام و منحنی Weibull
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(synthetic_data, bins=40, density=True, alpha=0.5, color='blue', 
        edgecolor='black', label='داده')

x = np.linspace(0, max(synthetic_data) * 1.1, 500)
params = {'shape': result['shape'], 'scale': result['scale']}
pdf = weibull_dist.pdf(x, params)
ax.plot(x, pdf, 'r-', linewidth=2.5, label=f'Weibull(shape={result["shape"]:.2f}, scale={result["scale"]:.2f})')

ax.axvline(shape_true, color='green', linestyle='--', linewidth=2, label=f'Shape واقعی: {shape_true}')
ax.axvline(scale_true, color='orange', linestyle='--', linewidth=2, label=f'Scale واقعی: {scale_true}')

ax.set_xlabel('مقدار', fontsize=12)
ax.set_ylabel('چگالی احتمال', fontsize=12)
ax.set_title('برازش توزیع Weibull روی داده مصنوعی', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ۳. بارگذاری پلاگین‌های موجود
# ============================================================================

# بارگذاری توزیع‌های موجود
existing_plugins = load_plugins()
print(f"✅ توزیع‌های موجود: {len(existing_plugins)}")
for code, dist in existing_plugins.items():
    print(f"   [{code}] {dist.name}")

# اضافه کردن توزیع Weibull به دیکشنری
all_plugins = existing_plugins.copy()
all_plugins[WeibullDistribution.code] = WeibullDistribution()
distributions = {dist.name: dist for dist in all_plugins.values()}

print(f"\n✅ تعداد کل توزیع‌ها پس از افزودن Weibull: {len(distributions)}")
print(f"   توزیع‌ها: {list(distributions.keys())}")

In [ ]:
# ============================================================================
# ۴. تست توزیع جدید روی داده نمونه
# ============================================================================

# بارگذاری داده نمونه
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values
data_year = data[:365, 1]  # tmean

# اطمینان از مثبت بودن داده (برای Weibull)
data_positive = data_year - np.min(data_year) + 0.1

# برازش Weibull روی داده
weibull_result = WeibullDistribution().fit(data_positive)

print("📈 نتایج برازش Weibull روی داده دمای نمونه:")
for key, value in weibull_result.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

In [ ]:
# ============================================================================
# ۵. مقایسه Weibull با سایر توزیع‌ها
# ============================================================================

all_results = {}
print("\n🔄 برازش همه توزیع‌ها (شامل Weibull)...")

for name, dist in distributions.items():
    try:
        if name == 'Weibull':
            res = dist.fit(data_positive)
        else:
            res = dist.fit(data_year)
        all_results[name] = res
        print(f"✅ {name}: AICc = {res.get('aicc', np.nan):.2f}")
    except Exception as e:
        print(f"❌ {name}: خطا - {str(e)}")
        all_results[name] = None

In [ ]:
# انتخاب بهترین مدل
valid_results = {k: v for k, v in all_results.items() 
                  if v is not None and 'aicc' in v and not np.isnan(v['aicc'])}

if valid_results:
    best_name = min(valid_results, key=lambda x: valid_results[x]['aicc'])
    print("=" * 60)
    print(f"🏆 بهترین توزیع: {best_name}")
    print(f"   AICc: {valid_results[best_name]['aicc']:.4f}")
    print("=" * 60)

# جدول مقایسه
comparison_df = pd.DataFrame([{
    'توزیع': name,
    'AICc': res['aicc'],
    'تعداد پارامتر': res.get('n_params', np.nan)
} for name, res in valid_results.items()])
comparison_df = comparison_df.sort_values('AICc').reset_index(drop=True)
comparison_df.index = comparison_df.index + 1
comparison_df

In [ ]:
# رسم مقایسه AICc با Weibull
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71' if name == best_name else '#e74c3c' if name == 'Weibull' else '#3498db' 
          for name in comparison_df['توزیع']]

bars = ax.barh(comparison_df['توزیع'], comparison_df['AICc'], 
               color=colors, alpha=0.7, edgecolor='black', linewidth=1)

ax.set_xlabel('AICc', fontsize=12)
ax.set_title('مقایسه AICc با توزیع Weibull', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, comparison_df['AICc']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}', 
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ۶. ارزیابی کیفیت Weibull
# ============================================================================

from core.quality.quality_flag import QualityFlag

flags = QualityFlag.evaluate(weibull_result, data_positive)

flag_names = {
    QualityFlag.PASS: '✅ PASS',
    QualityFlag.LOW_SAMPLE: '⚠️ LOW_SAMPLE',
    QualityFlag.NO_CONVERGENCE: '❌ NO_CONVERGENCE',
    QualityFlag.OUTLIER: '⚠️ OUTLIER',
    QualityFlag.HIGH_AICC: '⚠️ HIGH_AICC',
    QualityFlag.BAD_SKEW: '⚠️ BAD_SKEW',
    QualityFlag.NAN_INPUT: '❌ NAN_INPUT',
    QualityFlag.INF_INPUT: '❌ INF_INPUT',
}

print("📋 کیفیت برازش Weibull:")
for flag in flags:
    print(f"   {flag_names.get(flag, f'UNKNOWN ({flag})')}")

In [ ]:
# ============================================================================
# ۷. ذخیره توزیع جدید به عنوان فایل پلاگین
# ============================================================================

plugin_code = '''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
plugins/distributions/weibull.py
توزیع Weibull
"""

import numpy as np
from scipy import stats
from core.engine.distribution_plugin import DistributionPlugin

class WeibullDistribution(DistributionPlugin):
    name = "Weibull"
    code = 5
    params = ["shape", "scale"]
    n_params = 2
    supports_negative = False
    supports_zero = True
    supports_positive = True
    extreme_only = False

    def fit(self, data):
        data = np.array(data)[~np.isnan(data)]
        if len(data) < 3:
            return {
                'shape': np.nan,
                'scale': np.nan,
                'loglik': np.nan,
                'aicc': np.nan,
                'bic': np.nan
            }
        if np.any(data <= 0):
            data = data - np.min(data) + 1e-6
        try:
            shape, loc, scale = stats.weibull_min.fit(data, floc=0)
            loglik = np.sum(stats.weibull_min.logpdf(data, shape, loc=0, scale=scale))
            n = len(data)
            k = self.n_params
            aicc = -2*loglik + 2*k + (2*k*(k+1))/(n-k-1) if n > k+1 else np.inf
            bic = -2*loglik + k*np.log(n)
            return {
                'shape': shape,
                'scale': scale,
                'loglik': loglik,
                'aicc': aicc,
                'bic': bic
            }
        except Exception as e:
            return {
                'shape': np.nan,
                'scale': np.nan,
                'loglik': np.nan,
                'aicc': np.nan,
                'bic': np.nan
            }

    def pdf(self, x, params):
        shape = params.get('shape', 1.0)
        scale = params.get('scale', 1.0)
        return stats.weibull_min.pdf(x, shape, loc=0, scale=scale)
'''

plugin_dir = os.path.join(project_root, 'plugins', 'distributions')
os.makedirs(plugin_dir, exist_ok=True)

plugin_path = os.path.join(plugin_dir, 'weibull.py')
with open(plugin_path, 'w', encoding='utf-8') as f:
    f.write(plugin_code)

print(f"✅ فایل پلاگین Weibull در {plugin_path} ذخیره شد.")
print("\n📌 برای استفاده از توزیع Weibull در پروژه:")
print("   1. فایل weibull.py در پوشه plugins/distributions/ قرار دارد.")
print("   2. توزیع به‌صورت خودکار توسط load_plugins() شناسایی می‌شود.")
print("   3. می‌توانید از WeibullDistribution مستقیماً استفاده کنید.")

## 📋 جمع‌بندی

در این نوت‌بوک یاد گرفتید:

✅ معماری پلاگین توزیع‌ها در Climatology Engine
✅ ساختار کلاس `DistributionPlugin`
✅ پیاده‌سازی تابع `fit()` برای توزیع Weibull
✅ افزودن توزیع جدید به سیستم
✅ تست توزیع جدید روی داده مصنوعی و واقعی
✅ مقایسه توزیع جدید با سایر توزیع‌ها
✅ ارزیابی کیفیت برازش توزیع جدید
✅ ذخیره توزیع جدید به عنوان فایل پلاگین

---

**نکات کلیدی:**

1. هر توزیع جدید باید از `DistributionPlugin` ارث‌بری کند.
2. متد `fit()` باید دیکشنری با کلیدهای پارامترها، `loglik`، `aicc` و `bic` برگرداند.
3. کد توزیع باید منحصر‌به‌فرد باشد (از ۵ به بعد).
4. توزیع‌های جدید در پوشه `plugins/distributions/` قرار می‌گیرند.
5. سیستم به‌صورت خودکار پلاگین‌ها را شناسایی می‌کند.

---

**مراحل بعدی:**
- نوت‌بوک ۱۰: کاربرد پیشرفته